# Navigating with XPath

The real power of JNodes is that all 15 XPath axes work on them. If you know XPath from XML, you already know how to navigate JSON.

## Child Axis

Navigate to direct children by key name or wildcard:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(map {
    "title": "Hamlet",
    "author": "Shakespeare",
    "year": 1600
})
return (
    (: Named child step — like $element/title in XML :)
    "Title: " || fn:jvalue($tree/title),
    (: Wildcard — all children :)
    "All values: " || string-join(
        for $c in $tree/child::* return fn:jvalue($c), ", "
    )
)

## Descendant Axis

Reach deep into nested structures without spelling out every intermediate step:

In [ ]:
xquery version "4.0";

let $data := map {
    "store": map {
        "books": array {
            map { "title": "Hamlet", "price": 9.99 },
            map { "title": "Ulysses", "price": 14.99 },
            map { "title": "Moby Dick", "price": 7.99 }
        }
    }
}
let $tree := fn:jtree($data)

(: descendant::* finds all nodes at any depth :)
return (
    "Total nodes in tree: " || count($tree/descendant::*),
    "Titles: " || string-join(
        for $t in $tree/descendant::*/self::*[fn:jkey(.) = "title"]
        return fn:jvalue($t), ", "
    )
)

## Parent and Ancestor Axes

Navigate upward — find the context of a value:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(map {
    "database": map {
        "name": "eXist-db",
        "config": map { "port": 8080 }
    }
})
let $port := ($tree/descendant::*[fn:jkey(.) = "port"])[1]
return (
    "Port value: " || fn:jvalue($port),
    "Parent key: " || fn:jkey(fn:jparent($port)),
    "Ancestor count: " || count($port/ancestor::*)
)

## Sibling Axes

Navigate sideways among children of the same parent:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(map {
    "first": "Ada",
    "middle": "Augusta",
    "last": "Lovelace",
    "title": "Countess"
})
let $middle := ($tree/child::*[fn:jkey(.) = "middle"])[1]
return (
    "Node: " || fn:jkey($middle) || " = " || fn:jvalue($middle),
    "Following siblings: " || count($middle/following-sibling::*),
    "Preceding siblings: " || count($middle/preceding-sibling::*)
)

## Kind Tests

Filter nodes by their JSON type using kind tests:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(map {
    "name": "eXist-db",
    "version": 7,
    "stable": true(),
    "modules": array { "lucene", "ft" }
})
for $child in $tree/child::*
return fn:jkey($child) || ": " ||
    (if ($child instance of string-node()) then "string"
     else if ($child instance of number-node()) then "number"
     else if ($child instance of boolean-node()) then "boolean"
     else if ($child instance of array-node()) then "array"
     else if ($child instance of object-node()) then "object"
     else "other")

## Maps and Arrays as Path Operands

In XQuery 4.0, you can also use the `/` operator directly on maps and arrays, without calling [`fn:jtree()`]({docs}/functions/fn/jtree) first. Named steps look up keys; `*` returns all values:

In [ ]:
xquery version "4.0";

let $books := map {
    "fiction": array {
        map { "title": "Hamlet", "author": "Shakespeare" },
        map { "title": "Ulysses", "author": "Joyce" }
    },
    "reference": array {
        map { "title": "XQuery", "author": "Walmsley" }
    }
}
return map {
    "fiction count": count($books/fiction/*),
    "all authors": string-join(
        for $b in $books/*/child::* return $b?author, ", "
    )
}

This is a convenience — the engine navigates maps and arrays directly. For full axis support (parent, ancestor, sibling, following, preceding), convert to a JNode tree with [`fn:jtree()`]({docs}/functions/fn/jtree) first.